# 数据处理

#### 原始数据集本身没用数据进行处理

In [ ]:
import json
import re
import jsonlines

# 示例原始数据
with jsonlines.open('F:/Nijia.Han23/PycharmProjects/Medhall/MedHallu/Datasets/pqa_tcm/TCMP_HALLU.jsonl', 'r') as reader:
    data = [obj for obj in reader]
   

# # 处理前10条数据并打印清理结果
# for i in range(10):
#     original = data[i]['original_data']
#     # 清理 instruction 字段
#     if 'instruction' in original:
#         del original['instruction']
#     # 清理 question 字段
#     if 'question' in original:
#         original['question'] = original['question'].split('答案：')[0].split('\nA.')[0].strip()
#     # 清理 long_answer 字段
#     if 'long_answer' in original:
#         original['long_answer'] = re.sub(r'^答案：', '', original['long_answer']).strip()
#     # 清理 context 字段
#     if 'context' in original:
#         original['context'] = re.sub(r'^该问题解析：', '', original['context']).strip()
#     print(f"第{i+1}条数据：")
#     print(json.dumps(original, ensure_ascii=False, indent=4))
# 保存处理后的数据  
with jsonlines.open('F:/Nijia.Han23/PycharmProjects/Medhall/MedHallu/Datasets/pqa_tcm/demo.json', 'w') as writer:
    for item in data:
        original = item['original_data']
        # 清理 instruction 字段
        if 'instruction' in original:
            del original['instruction']
        # 清理 question 字段
        if 'question' in original:
            original['question'] = original['question'].split('答案：')[0].split('\nA.')[0].strip()
        # 清理 long_answer 字段
        if 'long_answer' in original:
            original['long_answer'] = re.sub(r'^答案：', '', original['long_answer']).strip()
        # 清理 context 字段
        if 'context' in original:
            original['context'] = re.sub(r'^该问题解析：', '', original['context']).strip()
        
        


第1条数据：
{
    "question": "百令片适用于____中医疾病",
    "long_answer": "A.咳嗽。",
    "context": "该方剂中医疾病为咯血，咳嗽，气喘",
    "options_list": [
        "A. 咳嗽",
        "B. 经闭",
        "C. 脱发症",
        "D. 流产",
        "E. 混合痔"
    ],
    "hallucination_answer": "D. 流产"
}
第2条数据：
{
    "question": "黄金波药酒具有____功效",
    "long_answer": "C.温中和胃。",
    "context": "该方剂功效为祛风活血，温中和胃",
    "options_list": [
        "A. 清疽解毒",
        "B. 润燥止痒",
        "C. 温中和胃",
        "D. 温宫止带",
        "E. 安神定志"
    ],
    "hallucination_answer": "E. 安神定志"
}
第3条数据：
{
    "question": "芩暴红止咳糖浆具有____功效",
    "long_answer": "A.止咳平喘。",
    "context": "该方剂功效为清热化痰，止咳平喘",
    "options_list": [
        "A. 止咳平喘",
        "B. 养血润燥",
        "C. 抗结核",
        "D. 排脓祛瘀",
        "E. 芳香解表"
    ],
    "hallucination_answer": "E. 芳香解表"
}
第4条数据：
{
    "question": "冰硼散中君药是____",
    "long_answer": "D.冰片。",
    "context": "该方剂君臣佐使为君药：冰片；臣药：硼砂；佐药：玄明粉、朱砂",
    "options_list": [
        "A. 岩白菜",
        "B. 巴戟天",
        "C. 千金子",
        "D

#### 问题数据改写

In [1]:
# 从原始数据集中提取100条数据作为demo - 'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\demo.json'
import os
import json
import jsonlines
from typing import Dict, Any

# 使用jsonlines读取demo_bak.json
with jsonlines.open(r'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\TCMP_HALLU.jsonl', 'r') as reader:
    texts = [obj for obj in reader]
    print(f"Loaded {len(texts)} items from the dataset after saving.")

# 提取前10000条数据
demo_texts = texts[:10000]
# 保存为新的JSON文件
with open(r'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\demo.json', 'w', encoding='utf-8') as f:
    json.dump(demo_texts, f, ensure_ascii=False, indent=4)
    print(f"Saved {len(demo_texts)} items to the demo dataset.")


Loaded 65867 items from the dataset after saving.
Saved 10000 items to the demo dataset.


In [ ]:
# 利用大模型进行问句格式转换
from openai import OpenAI
import json
from typing import Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

client = OpenAI()

def generate_question_type(question:str) -> Dict[str, Any]:
        
        prompt = (
            "请将以下填空题转换为适合中医药领域的问句格式：\n"
            f"问题：{question}\n"
            "请返回一个JSON对象，包含以下字段：\n"
            "- 'questionAsk': 转换后的问题\n"
            
        )
    
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"请分析以下文本：\n{question}"}
            ],
            temperature=0.2,
            max_tokens=500,
            response_format={"type": "json_object"}
        )

        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except json.JSONDecodeError:
            return {"error": "Invalid JSON response"}
        
# 处理函数，将每个问题传递给生成函数并返回结果
def process_item(item):
    question = item.get('question', '')
    result = generate_question_type(question)
    item['questionAsk'] = result.get('questionAsk', question)
    return item

# 示例：加载demo_texts文件
demo_texts = json.load(open(r'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\demo_gpt.json', 'r', encoding='utf-8'))

# 使用ThreadPoolExecutor并行处理问题
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_item, item) for item in demo_texts[:10000]]
    # 进度条显示
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing questions"):
        future.result()  # 获取结果，确保所有任务完成

# 保存转换后的结果
with open(r'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\demo_gpt4.1.json', 'w', encoding='utf-8') as f:
    json.dump(demo_texts, f, ensure_ascii=False, indent=4)
    print(f"Saved {len(demo_texts)} items to the demo dataset after question format conversion.")

Processing questions: 100%|██████████| 10000/10000 [32:06<00:00,  5.19it/s] 

Saved 10000 items to the demo dataset after question format conversion.


In [ ]:
# QWen3-30B-A3B 

import torch
import numpy as np
import random
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import json
from torch.utils.data import DataLoader
from typing import Dict, Any
from openai import OpenAI
from zhipuai import ZhipuAI
import time

# 调用API来进行推理
client = OpenAI()

system_prompt = """You are an AI assistant with extensive knowledge in the field of medicine. I want you act as an answer judge.
Given a question and an answer, your objective is to determine if the provided answer contains non-factual or hallucinated information. 
You SHOULD give your judgement based on the following the world knowledge.
You are trying to determine if there is a factual contradiction between the answer and the world knowledge. Some information in the answer might be fabricated.
Answer '0' if the answer is factual and '1' if the answer is hallucinated. Please return the result in JSON format. The output must be a JSON object. Include ‘result'. """

user_prompt = """ 
World Knowledge:{knowledge}
Question:{question}
Answer:{answer} Return '0' if the answer is factual and '1' if the answer is hallucinated. No explanation is needed."""

with open(r'F:\Nijia.Han23\PycharmProjects\Medhall\MedHallu\Datasets\pqa_tcm\demo_gpt.json') as f:
    train_data = json.load(f)

results = []


# 统计所返回的数目中1占比
for item in train_data:
    question = item.get('questionAsk', '')
    answer = item.get('ground_truth_expand', '')
    context = item.get('context', '')
    search_results = item.get('search_results', '')
    formatted_user_prompt = user_prompt.format(knowledge=search_results,question=question, answer=answer)

    # 重试 3 次
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="qwen3-32b",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": formatted_user_prompt}
                ],
                temperature=0.6,
                max_tokens=100,
                stream=True
            )

            for chunk in response:
                if not chunk.choices:
                    continue
                delta = chunk.choices[0].delta.content
                if delta:
                    print(delta, end="", flush=True)
                time.sleep(0.01)     # 限速
            print()                  # 换行
            break                    # 成功就跳出重试

        except Exception as e:
            rid = getattr(e, "response", {}).headers.get("x-request-id", "unknown") if hasattr(e, "response") else "unknown"
            print(f"[WARN] attempt {attempt + 1}/3  request-id={rid}  {e}")
            if attempt < 2:
                time.sleep(2 ** attempt)


# # 计算1的占比
# num_1 = sum(1 for r in results if str(r) == '1')
# num_valid = sum(1 for r in results if r is not None)
# ratio = num_1 / num_valid if num_valid > 0 else 0
# print(f"1的占比: {ratio:.2f}")

```json
{
  "result": "1"
}
```
```json
{"result": 0}
```
{"result": 0}
```json
{
  "result": 0
}
```
{"result": 0}
```json
{
  "result": 1
}
```
{"result": 0}
```json
{"result": 1}
```
```json
{
  "result": "1"
}
```
```json
{"result": 1}
```
{"result": "1"}
[WARN] attempt 1/3  request-id=unknown  peer closed connection without sending complete message body (incomplete chunked read)
{"result": "1"}
{"result": "1"}
{"result": 0}
{"result": "1"}
```json
{"result": 0}
```
```json
{
  "result": 0
}
```
{"result": 0}
{"result": 0}
{"result": 0}
```json
{
  "result": 1
}
```
{"result": 0}
```json
{
  "result": 1
}
```
{"result": 0}
{"result": "1"}
{"result": "0"}
{"result": "0"}
{"result": 1}
{"result": "1"}
{"result": "1"}
{"result": 1}
{"result": "1"}
1
{"result": "0"}
{"result": "1"}
{"result": "0"}
```json
{"result": "0"}
```
{"result": "0"}
{"result": "1"}
```json
{
  "result": 1
}
```
{"result": 0}
```json
{"result": "1"}
```
```json
{
  "result": 1
}
```
{"result": 0}
{"result": "1"}

KeyboardInterrupt: 